# QDGrasp Phase 0 CUDA hardware smoke

Fail-closed CUDA validation. QDGrasp is installed from an exact public commit; no source copy, dataset, checkpoint, credential, or machine-specific path is embedded.

In [ ]:
import subprocess
import sys

assert sys.version_info >= (3, 11), f'Python >=3.11 required, got {sys.version}'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'torch==2.11.0+cu128',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'lightning==2.6.5', 'mujoco==3.12.0', 'numpy==2.4.6',
    'scipy==1.17.1', 'trimesh==4.12.2', 'safetensors==0.8.0',
    'pydantic==2.13.4', 'PyYAML==6.0.3', 'einops==0.8.2',
    'rich==14.3.4', 'typer==0.27.1',
], check=True)
QDGRASP_COMMIT = '5cc088580f16e7f6c8184981e85183c9db1ae4c6'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
    f'git+https://github.com/ninicom/qdgrasp.git@{QDGRASP_COMMIT}',
], check=True)
print(f'QDGrasp {QDGRASP_COMMIT} installed on Python {sys.version.split()[0]}')

In [ ]:
from pathlib import Path

smoke_source = r'''
import hashlib
import importlib.metadata as metadata
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import lightning.fabric
import mujoco
import qdgrasp
import torch

QDGRASP_COMMIT = '5cc088580f16e7f6c8184981e85183c9db1ae4c6'
assert torch.__version__.startswith('2.11.0+cu128'), torch.__version__
assert torch.version.cuda == '12.8', torch.version.cuda
device = qdgrasp.require_cuda(expected_runtime='12.8')
properties = torch.cuda.get_device_properties(device)
assert torch.cuda.get_device_capability(device) >= (7, 5), properties.name

model = torch.nn.Sequential(
    torch.nn.Linear(64, 128), torch.nn.GELU(), torch.nn.Linear(128, 32)
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler('cuda')
torch.manual_seed(20260822)
torch.cuda.manual_seed_all(20260822)
losses = []
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    x = torch.randn(32, 64, device=device)
    target = torch.randn(32, 32, device=device)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        loss = torch.nn.functional.mse_loss(model(x), target)
    assert torch.isfinite(loss), loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    losses.append(float(loss.detach().cpu()))
torch.cuda.synchronize()

output_dir = Path.cwd()
checkpoint_path = output_dir / 'smoke_checkpoint.pt'
torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict()}, checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model'], strict=True)
optimizer.load_state_dict(checkpoint['optimizer'])

mj_model = mujoco.MjModel.from_xml_string(
    "<mujoco><worldbody><body><joint/><geom type='sphere' size='.01'/></body></worldbody></mujoco>"
)
mj_data = mujoco.MjData(mj_model)
mujoco.mj_forward(mj_model, mj_data)

evidence = {
    'schema_version': 1,
    'status': 'pass',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': properties.name,
    'device_capability': list(torch.cuda.get_device_capability(device)),
    'device_memory_bytes': properties.total_memory,
    'lightning': metadata.version('lightning'),
    'mujoco': metadata.version('mujoco'),
    'numpy': metadata.version('numpy'),
    'qdgrasp': qdgrasp.__version__,
    'qdgrasp_commit': QDGRASP_COMMIT,
    'qdgrasp_module': qdgrasp.__name__,
    'amp_train_steps': len(losses),
    'losses': losses,
    'checkpoint_resume': 'pass',
    'mujoco_forward': 'pass',
}
payload = json.dumps(evidence, indent=2, sort_keys=True) + '\n'
evidence['payload_sha256'] = hashlib.sha256(payload.encode()).hexdigest()
evidence_path = output_dir / 'phase0_cuda_evidence.json'
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
print(json.dumps(evidence, indent=2, sort_keys=True))
'''
script_path = Path.cwd() / 'phase0_cuda_smoke.py'
script_path.write_text(smoke_source)
subprocess.run([sys.executable, str(script_path)], check=True)